In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table(
    "workspace.default.freddie_model_base"
)

# Reproducible random ordering within event types
w_order = (
    Window.partitionBy("event_type")
    .orderBy(
        F.xxhash64(
            F.col("loan_id"),
            F.lit("freddie_2017_v1")
        ),
        F.col("loan_id")
    )
)

w_group = Window.partitionBy("event_type")

df_split = (
    df
    .withColumn("_rank", F.row_number().over(w_order))
    .withColumn("_group_n", F.count("*").over(w_group))
    .withColumn(
        "split",
        F.when(
            F.col("_rank") <= F.floor(F.col("_group_n") * 0.70),
            "train"
        )
        .when(
            F.col("_rank") <= F.floor(F.col("_group_n") * 0.85),
            "validation"
        )
        .otherwise("test")
    )
    .drop("_rank", "_group_n")
)


In [0]:

display(
    df_split.groupBy("split")
    .agg(
        F.count("*").alias("n_loans"),
        F.sum("event").alias("n_events"),
        F.round(
            100 * F.avg("event"), 2
        ).alias("event_pct")
    )
    .orderBy("split")
)

# Event type balance across splits
display(
    df_split.groupBy("split", "event_type")
    .count()
    .orderBy("split", "event_type")
)

# Verify mutually exclusive loan assignment
display(
    df_split.groupBy("loan_id")
    .agg(F.countDistinct("split").alias("n_splits"))
    .filter(F.col("n_splits") != 1)
)


split,n_loans,n_events,event_pct
test,7498,440,5.87
train,34975,2048,5.86
validation,7495,439,5.86


split,event_type,count
test,0,1625
test,1,440
test,2,5420
test,3,13
train,0,7580
train,1,2048
train,2,25289
train,3,58
validation,0,1625
validation,1,439


loan_id,n_splits


In [0]:

df_split.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.freddie_model_split"
    )

print("Train/validation/test split saved!")


Train/validation/test split saved!
